# NLLB-200 Sinhala Subtitle Fine-tuning

Clean Colab workflow. Run top to bottom. This notebook does not overwrite project scripts.

## Cell 1 - Check GPU

In [ ]:
import torch

print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print(torch.cuda.get_device_name(0))
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")


## Cell 2 - Mount Google Drive and Set Paths

In [ ]:
from google.colab import drive

try:
    drive.flush_and_unmount()
except Exception as exc:
    print('Drive not mounted, so nothing to flush and unmount.')

drive.mount('/content/drive', force_remount=True)

DRIVE_BASE = '/content/drive/MyDrive/sinhala-subtitle'
SCRIPTS_DIR = f'{DRIVE_BASE}/sinhala-finetune'
DATA_FILE = f'{DRIVE_BASE}/clean_pairs_final.tsv'
CHECKPOINT_DIR = f'{DRIVE_BASE}/checkpoints'

LOCAL_SCRIPTS = '/content/sinhala-finetune'
LOCAL_DATA = '/content/clean_pairs_final.tsv'

print('DRIVE_BASE:', DRIVE_BASE)
print('SCRIPTS_DIR:', SCRIPTS_DIR)
print('DATA_FILE:', DATA_FILE)
print('CHECKPOINT_DIR:', CHECKPOINT_DIR)


## Cell 3 - Install Dependencies

In [ ]:
!pip install -q transformers datasets accelerate sentencepiece sacrebleu evaluate pandas

import transformers, datasets, accelerate
print('transformers', transformers.__version__)
print('datasets', datasets.__version__)
print('accelerate', accelerate.__version__)


## Cell 4 - Copy Project Files into Colab Runtime

In [ ]:
import os
import shutil
import pandas as pd

if os.path.exists(LOCAL_SCRIPTS):
    shutil.rmtree(LOCAL_SCRIPTS)

shutil.copytree(
    SCRIPTS_DIR,
    LOCAL_SCRIPTS,
    ignore=shutil.ignore_patterns('*.gdoc', '*.gsheet', '*.gslides', '__pycache__')
)
shutil.copy2(DATA_FILE, LOCAL_DATA)

print(f'Scripts copied to {LOCAL_SCRIPTS}')
print(f'Data copied to {LOCAL_DATA}')
print('Scripts:', sorted(os.listdir(LOCAL_SCRIPTS)))

required = {'train.py', 'dataset.py', 'evaluate.py', 'requirements.txt'}
missing = required - set(os.listdir(LOCAL_SCRIPTS))
if missing:
    raise FileNotFoundError(f'Missing required script(s): {sorted(missing)}')

df = pd.read_csv(LOCAL_DATA, sep='\t', dtype=str).dropna()
print(f'\nDataset loaded: {len(df):,} pairs')
print('\nFirst 3 rows:')
for _, row in df.head(3).iterrows():
    print(f'  EN: {row["source"]}')
    print(f'  SI: {row["target"]}')
    print()


## Cell 5 - Smoke Test

In [ ]:
import subprocess
import sys

BATCH = 8

smoke_cmd = [
    sys.executable, '/content/sinhala-finetune/train.py',
    '--data', '/content/clean_pairs_final.tsv',
    '--output-dir', '/content/smoke_test',
    '--epochs', '1',
    '--batch-size', str(BATCH),
    '--grad-accum', '1',
    '--sample', '100',
    '--save-steps', '999999',
    '--eval-steps', '999999',
    '--num-proc', '1',
    '--fp16',
]

print('Running smoke test...\n')
print(' '.join(smoke_cmd), '\n')
result = subprocess.run(
    smoke_cmd,
    cwd='/content/sinhala-finetune',
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
)
print(result.stdout)

if result.returncode != 0:
    raise RuntimeError(f'Smoke test failed with exit code {result.returncode}')
print('\nSmoke test PASSED.')


## Cell 6 - Full Training

In [ ]:
import os
import subprocess
import sys

os.makedirs(CHECKPOINT_DIR, exist_ok=True)
BATCH = 8

train_cmd = [
    sys.executable, '/content/sinhala-finetune/train.py',
    '--data', LOCAL_DATA,
    '--output-dir', CHECKPOINT_DIR,
    '--epochs', '3',
    '--batch-size', str(BATCH),
    '--grad-accum', '4',
    '--lr', '2e-5',
    '--warmup-ratio', '0.03',
    '--save-steps', '500',
    '--eval-steps', '500',
    '--logging-steps', '100',
    '--num-proc', '1',
    '--fp16',
]

print('Starting full training...')
print('Checkpoints will save to:', CHECKPOINT_DIR)
print(' '.join(train_cmd), '\n')

result = subprocess.run(train_cmd, cwd='/content/sinhala-finetune')
if result.returncode != 0:
    raise RuntimeError(f'Training exited with code {result.returncode}')
print('\nTraining complete!')


## Cell 7 - Resume Training After Disconnect

In [ ]:
import os
import subprocess
import sys

os.makedirs(CHECKPOINT_DIR, exist_ok=True)

resume_cmd = [
    sys.executable, '/content/sinhala-finetune/train.py',
    '--data', LOCAL_DATA,
    '--output-dir', CHECKPOINT_DIR,
    '--epochs', '3',
    '--batch-size', '8',
    '--grad-accum', '4',
    '--lr', '2e-5',
    '--warmup-ratio', '0.03',
    '--save-steps', '500',
    '--eval-steps', '500',
    '--logging-steps', '100',
    '--num-proc', '1',
    '--fp16',
    '--resume',
]

print('Resuming training...')
print(' '.join(resume_cmd), '\n')
result = subprocess.run(resume_cmd, cwd='/content/sinhala-finetune')
if result.returncode != 0:
    raise RuntimeError(f'Resume exited with code {result.returncode}')


## Cell 8 - Quick Translation Test

In [ ]:
import glob
import torch
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer

checkpoints = sorted(glob.glob(f'{CHECKPOINT_DIR}/**/final', recursive=True))
if not checkpoints:
    checkpoints = sorted(glob.glob(f'{CHECKPOINT_DIR}/checkpoint-*'))
if not checkpoints:
    raise FileNotFoundError('No checkpoint found yet.')

model_path = checkpoints[-1]
print('Using model:', model_path)

tokenizer = AutoTokenizer.from_pretrained(model_path)
model = AutoModelForSeq2SeqLM.from_pretrained(model_path).to('cuda' if torch.cuda.is_available() else 'cpu')
tokenizer.src_lang = 'eng_Latn'
forced_bos_token_id = tokenizer.convert_tokens_to_ids('sin_Sinh')

sentences = [
    'I need to talk to you.',
    'Where are you going?',
    'This is not a game.',
]

inputs = tokenizer(sentences, return_tensors='pt', padding=True, truncation=True).to(model.device)
with torch.no_grad():
    outputs = model.generate(**inputs, forced_bos_token_id=forced_bos_token_id, max_new_tokens=80)

for src, pred in zip(sentences, tokenizer.batch_decode(outputs, skip_special_tokens=True)):
    print('EN:', src)
    print('SI:', pred)
    print()


## Cell 9 - BLEU Sample

In [ ]:
import subprocess
import sys

checkpoints = sorted(glob.glob(f'{CHECKPOINT_DIR}/**/final', recursive=True))
model_path = checkpoints[-1] if checkpoints else sorted(glob.glob(f'{CHECKPOINT_DIR}/checkpoint-*'))[-1]

eval_cmd = [
    sys.executable, '/content/sinhala-finetune/evaluate.py',
    '--data', LOCAL_DATA,
    '--model', model_path,
    '--sample', '1000',
    '--batch-size', '8',
]
print(' '.join(eval_cmd))
result = subprocess.run(eval_cmd, cwd='/content/sinhala-finetune')
if result.returncode != 0:
    raise RuntimeError(f'Evaluation exited with code {result.returncode}')


## Cell 10 - Notes

For SRT translation, add a dedicated script after the base fine-tune is working.